# Run stages

Parameter work is staged: **compile** (host), **run start** (`prepare_fn` +
hoisted rate subtrees), and **per step** (vector field). See
`docs/dev/run-stages.md`.

The trajectory is `prepare_fn` deriving `beta = r0 / period` and a
model that was given that beta directly — the lines should overlie.
The knot-count bars are the loop-body cost: with hoisting, K = 4 and
K = 40 should be the same height; without it, 40 should be taller.
The last bars are the hoist table: parameter-only arithmetic is slotted in
it, and a deferred function reading `y` and `t` beside it adds no entries.
`derived_fn` was removed in step 41; `field_refs(defer(f)(...), Schema)`
replaces it.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from collections import Counter
from typing import Any, NamedTuple

import numpy as np
import jax
import jax.numpy as jnp

from summer4 import (
    Compartments,
    EntryFlow,
    FlowModel,
    Param,
    Params,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    StateArray,
    Time,
    defer,
    field_refs,
)
from summer4.timevarying import linear


## Stage table

| Stage | When | Produces |
|---|---|---|
| 0 compile | once per structure | index arrays, hoist table |
| 1 run start | once per `run` | `Prepared`, initial state |
| 2 per step | every VF call | rates reading hoisted values |

## `prepare_fn` computes a derived constant

The caller passes `r0` and `period`. Before the solve, `prepare_fn`
stores `beta = r0 / period`. An entry at that beta should draw the
same line as passing `beta` yourself.


In [ ]:
state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
model = FlowModel(pmap)
model.add_flow(EntryFlow("in", state["Y"], Param("beta")))

def prepare_fn(p):
    return {**p, "beta": p["r0"] / p["period"]}

cm = model.compile(prepare_fn=prepare_fn)
y0 = PropertyData.wrap(pmap, np.array([0.0]))
plan = SavePlan(requests={"y": SaveRequest(Compartments())}, ts=np.array([1.0]))
res = cm.run({"r0": 2.0, "period": 4.0}, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
bare = FlowModel(pmap)
bare.add_flow(EntryFlow("in", state["Y"], Param("beta")))
res_b = bare.compile().run({"beta": 0.5}, y0, t0=0.0, t1=1.0, dt=0.25, save=plan)
np.testing.assert_allclose(
    np.asarray(res["y"].values.data), np.asarray(res_b["y"].values.data)
)

plot_ts = np.linspace(0.0, 1.0, 21)
plot_plan = SavePlan(requests={"y": SaveRequest(Compartments())}, ts=plot_ts)
via_prepare = cm.run(
    {"r0": 2.0, "period": 4.0}, y0, t0=0.0, t1=1.0, dt=0.05, save=plot_plan
)
via_beta = bare.compile().run(
    {"beta": 0.5}, y0, t0=0.0, t1=1.0, dt=0.05, save=plot_plan
)
pd.DataFrame(
    {
        "prepare_fn (r0 / period)": np.asarray(via_prepare["y"].values.data)[:, 0],
        "beta = 0.5 directly": np.asarray(via_beta["y"].values.data)[:, 0],
    },
    index=plot_ts,
).plot(
    title="prepare_fn derives beta before the first step",
    labels={"index": "time", "value": "people"},
)


## Interp knots are hoisted (loop-body size)

With `hoist=True`, stacking `K` FieldRef knots happens at run start, so the
scan/while body does not grow with `K`. Hoisting on should be flat
from 4 knots to 40; hoisting off should grow.


In [ ]:
def _sub_jaxprs(eqn):
    out = []
    for v in eqn.params.values():
        for item in (v if isinstance(v, (tuple, list)) else (v,)):
            if hasattr(item, "jaxpr") and hasattr(item.jaxpr, "eqns"):
                out.append(item.jaxpr)
            elif hasattr(item, "eqns"):
                out.append(item)
    return out


def _all_eqns(jaxpr):
    for eqn in jaxpr.eqns:
        yield eqn
        for sub in _sub_jaxprs(eqn):
            yield from _all_eqns(sub)


def loop_body_primitives(closed):
    counts = Counter()
    jaxpr = closed.jaxpr if hasattr(closed, "jaxpr") else closed
    for eqn in jaxpr.eqns:
        if eqn.primitive.name in {"scan", "while"}:
            for sub in _sub_jaxprs(eqn):
                for inner in _all_eqns(sub):
                    counts[inner.primitive.name] += 1
        else:
            for sub in _sub_jaxprs(eqn):
                counts.update(loop_body_primitives(sub))
    return counts


def loop_body_stack_widths(closed):
    """In-loop stack widths; JAX 0.11+ keeps one stack of width K."""
    widths = []
    jaxpr = closed.jaxpr if hasattr(closed, "jaxpr") else closed

    def walk(jp, *, inside_loop=False):
        for eqn in jp.eqns:
            is_loop = eqn.primitive.name in {"scan", "while"}
            if inside_loop and eqn.primitive.name == "stack":
                widths.append(len(eqn.invars))
            for sub in _sub_jaxprs(eqn):
                walk(sub, inside_loop=inside_loop or is_loop)

    walk(jaxpr)
    return widths


def loop_knot_cost(closed):
    # JAX 0.11+: max stack width in the euler scan. JAX 0.6: unrolled eqn count.
    widths = loop_body_stack_widths(closed)
    if widths:
        return max(widths)
    return int(sum(loop_body_primitives(closed).values()))


def body_cost(k, *, hoist):
    Knots = NamedTuple("Knots", [(f"v{i}", float) for i in range(k)])
    refs = field_refs(Params(), Knots)
    expr = linear(Time(), tuple(float(i) for i in range(k)), tuple(getattr(refs, f"v{i}") for i in range(k)))
    model = FlowModel(pmap)
    model.add_flow(EntryFlow("in", state["Y"], expr))
    cm = model.compile(hoist=hoist)
    params = Knots(**{f"v{i}": float(i % 3) for i in range(k)})

    def loss(p):
        r = cm.run(p, y0, t0=0.0, t1=1.0, dt=0.25, save=plan, solver="euler")
        return jnp.sum(jnp.asarray(r["y"].values.data))

    return loop_knot_cost(jax.make_jaxpr(loss)(params))


on4, on40 = body_cost(4, hoist=True), body_cost(40, hoist=True)
off4, off40 = body_cost(4, hoist=False), body_cost(40, hoist=False)
print({"hoist_on": (on4, on40), "hoist_off": (off4, off40)})
assert on4 == on40
assert off40 > off4

pd.DataFrame(
    {"hoist on": [on4, on40], "hoist off": [off4, off40]},
    index=["K = 4", "K = 40"],
).plot.bar(
    title="Hoisting keeps the loop body the same size as the knot count grows",
    labels={"index": "knot count", "value": "in-loop stack width or eqn count"},
)


## Deferred code reading `y` and `t` does not disable hoisting

A `FieldRef` is always run stage. To compute named values from `t` and `y`
as well as the parameters, wrap a function with `defer` and read its result
through `field_refs`: `field_refs(defer(f)(Params(), y=StateArray(), t=Time()), Schema)`.
`f(params, *, y, t)` runs once per vector-field call however many rates read
the result. The `derived_fn` argument to `compile` did this job; it was
removed in step 41, together with the rule that made every `FieldRef`
step-stage.

Here `f` scales `b` by the population, so the deferred refs are step stage,
while `a * b` built from `field_refs(Params(), P)` stays run stage. The bars
count hoist-table entries: the deferred function adds none, and the
parameter-only product is still slotted.


In [ ]:
class P(NamedTuple):
    a: float
    b: float


def f(p: P, *, y: jnp.ndarray, t: jnp.ndarray) -> P:
    del t
    return P(a=p.a, b=p.b * jnp.sum(y))


pure = field_refs(Params(), P)
D = field_refs(defer(f)(Params(), y=StateArray(), t=Time()), P)
params = P(a=2.0, b=3.0)
y1 = PropertyData.wrap(pmap, np.array([2.0]))


def build(rate: Any) -> Any:
    m = FlowModel(pmap)
    m.add_flow(EntryFlow("in", state["Y"], rate))
    return m


m_pure = build(pure.a * pure.b + 1.0)
m_mixed = build(pure.a * pure.b + D.a * D.b)
cm_pure, cm_mixed = m_pure.compile(hoist=True), m_mixed.compile(hoist=True)
cm_mixed_off = m_mixed.compile(hoist=False)
r_on = cm_mixed.run(params, y1, t0=0.0, t1=1.0, dt=0.25, save=plan)
r_off = cm_mixed_off.run(params, y1, t0=0.0, t1=1.0, dt=0.25, save=plan)
np.testing.assert_allclose(
    np.asarray(r_on["y"].values.data), np.asarray(r_off["y"].values.data), rtol=1e-12
)

from summer4.flows.stages import build_hoist_table, roots_of

pure_table = build_hoist_table(roots_of(cm_pure.flows, cm_pure.order))
mixed_table = build_hoist_table(roots_of(cm_mixed.flows, cm_mixed.order))
assert len(pure_table.entries) >= 1
assert len(mixed_table.entries) == len(pure_table.entries)
pd.Series(
    {
        "parameter-only a * b": len(pure_table.entries),
        "with deferred f beside it": len(mixed_table.entries),
    }
).to_frame("entries").plot.bar(
    title="A deferred function reading y and t adds no hoist-table entries",
    labels={"index": "model", "value": "hoist-table entries"},
)
